# Assignment 3: Fine-tuning language models

In this assignment, you will perform supervised fine-tuning (SFT) of a small open LLM on an instruction tuning dataset. You will convert this dataset into instruction-response pairs, fine-tune a causal language model using LoRA (Low-Rank Adaptation), and evaluate it through prompted inference and comparison with other methods.

## Preliminaries

First, let's install the required libraries. If you are running in your own environment, make sure the following are installed:

- [Torch](https://docs.pytorch.org/docs/stable/index.html)
- [Transformers](https://huggingface.co/docs/transformers/index)
- [Datasets](https://huggingface.co/docs/datasets/index)
- [Evaluate](https://huggingface.co/docs/evaluate/en/index)
- [NLTK](https://www.nltk.org/api/nltk.html)
- [rouge_score](https://pypi.org/project/rouge-score/)

In a Colab notebook, most of them are already installed, except Evaluate and rouge_score.

In [2]:
%pip install evaluate rouge_score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24988 sha256=4ebdf2fc5bf5523b06a2f25b2a87878278abc449ad1a7e010d9b1b05a84f253b
  Stored in directory: /Users/ewishei/Library/Caches/pip/wheels/44/af/da/5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge_score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [evaluate]

[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


We also set some configuration parameters.

Most importantly, you should select a language model to work with in this assignment and enter its HuggingFace identifier in the parameter `MODEL_NAME` below. In principle you can use any model that you want, but we recommend that you select a model that has not already been trained to follow instructions, so it should be a "pure" language model trained on raw text (similar to Assignments 1 and 2).

The selected model should be small enough to fit in your computational environment. We have verified that the 135-million parameter [`SmolLM2` model](https://huggingface.co/HuggingFaceTB/SmolLM2-135M), developed by HuggingFace, can be used to solve this assignment in a Colab notebook (free tier, T4 GPU). If you run on a cluster, you can select a larger model (and probably see more interesting results).

We also define training and test set sizes here. Again, the values below have been set so that the assignment can be solved in Colab, and you can increase these sizes to improve the quality of the fine-tuned models.

In [3]:
import torch
SEED = 101

# Device selection: prefer CUDA, then Apple Silicon MPS, then CPU.
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print(f"Using device: {DEVICE}")

# bf16 only works reliably on Ampere+ NVIDIA GPUs. On MPS/CPU we disable it.
USE_BF16 = (DEVICE == "cuda")

MAX_TRAIN_SAMPLES = 5000
MAX_TEST_SAMPLES = 400

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M"
# The skeleton later references `model_name_or_path`; alias it here.
model_name_or_path = MODEL_NAME

torch.manual_seed(SEED)


Using device: mps


# Part 1: Preprocessing

### ⚙&nbsp; Task 1.1: Loading and inspecting the dataset

The dataset [SmolTalk](https://huggingface.co/datasets/HuggingFaceTB/smoltalk) is a collection of instruction-response pairs designed for SFT of large language models for instruction following. This dataset consists of examples of user inputs with system responses.

You can load using the datasets from the HuggingFace repository as follows.

In [4]:
from datasets import load_dataset
from datasets import DatasetDict

smoltalk = load_dataset("HuggingFaceTB/smoltalk", 'all')

README.md: 0.00B [00:00, ?B/s]

data/all/train-00000-of-00009.parquet:   0%|          | 0.00/223M [00:00<?, ?B/s]

data/all/train-00001-of-00009.parquet:   0%|          | 0.00/223M [00:00<?, ?B/s]

data/all/train-00002-of-00009.parquet:   0%|          | 0.00/223M [00:00<?, ?B/s]

data/all/train-00003-of-00009.parquet:   0%|          | 0.00/223M [00:00<?, ?B/s]

...

data/all/train-00007-of-00009.parquet:   0%|          | 0.00/224M [00:00<?, ?B/s]

data/all/train-00008-of-00009.parquet:   0%|          | 0.00/224M [00:00<?, ?B/s]

data/all/test-00000-of-00001.parquet:   0%|          | 0.00/105M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1043917 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/54948 [00:00<?, ? examples/s]

In order to make this assignment possible to solve in a restricted environment, we simplify the dataset a bit:
- We remove multi-turn chat dialogues from the dataset;
- We remove instances where the query or the answer is greater than a set maximum length;
- We keep a subset of the data for training and testing (by default 5000 and 400, respectively).

In [5]:
smoltalk_simplified = smoltalk.filter(lambda row: len(row['messages']) <= 3 and all(len(m['content']) <= 256 for m in row['messages']))
smoltalk_simplified = DatasetDict({
    "train": smoltalk_simplified["train"].select(range(MAX_TRAIN_SAMPLES)),
    "test": smoltalk_simplified["test"].select(range(MAX_TEST_SAMPLES)),
})

Filter:   0%|          | 0/1043917 [00:00<?, ? examples/s]

Filter:   0%|          | 0/54948 [00:00<?, ? examples/s]

In [6]:
smoltalk_simplified

DatasetDict({
    train: Dataset({
        features: ['messages', 'source'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['messages', 'source'],
        num_rows: 400
    })
})

Print some examples from the dataset so that you understand the format.

Key points you need to note here: each example from the training or test set consists of a sequence of messages. The number of messages in each example will be 2 or 3, because we removed multi-turn chat dialogues in the previous step. Each message is associated with a `role` label:
- `user`: an example of something the user might write.
- `assistant`: an example of an output an LLM could be expected to produce, given the input.
- `system`: a *system prompt* that gives guidelines for the general behavior of the LLM's behavior.

All examples in the dataset include a user input and an assistant output, but the system prompt is not available in all of the examples.

In [7]:
smoltalk_simplified['train'][0]

{'messages': [{'content': "You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.",
   'role': 'system'},
  {'content': 'Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.',
   'role': 'user'},
  {'content': 'The chef made more food after the restaurant ran out.',
   'role': 'assistant'}],
 'source': 'explore-instruct-rewriting'}

### 🎓&nbsp; Task 1.2: Formatting the data for instruction tuning

Define a function `format_input_output` that converts an example from the dataset into an input/output pair that we can use to fine-tune the LLM.

You are free to design the format. The following document gives some examples that have been used by different instruction-following LLMs including Llama and Mistral: https://huggingface.co/learn/llm-course/chapter11/2#common-template-formats

The later stages of our preprocessing pipeline expect that this function returns an object containing two parts: the `prompt` (what goes into the LLM before generating anything) and the `response` (what the LLM is expected to generate).

In [8]:
def format_input_output(example):
    """Convert a SmolTalk example into a (prompt, response) pair using ChatML.

    SmolLM2's tokenizer already includes the special tokens
    <|im_start|>, <|im_end|>, so we use the ChatML format here. The prompt
    contains the optional system message + the user turn and ends with
    "<|im_start|>assistant\n", which is exactly the position where the model
    should start generating. The response is the assistant message followed by
    <|im_end|> so the model also learns when to stop.
    """
    messages = example["messages"]

    # Split the trailing assistant turn from everything that comes before it.
    *context, assistant_msg = messages
    assert assistant_msg["role"] == "assistant", (
        f"Expected last message to be assistant, got {assistant_msg['role']}"
    )

    parts = []
    for m in context:
        parts.append(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n")
    # Open the assistant turn but DO NOT include its content yet — that goes in `response`.
    parts.append("<|im_start|>assistant\n")
    prompt = "".join(parts)

    response = f"{assistant_msg['content']}<|im_end|>\n"

    return {"prompt": prompt, "response": response}


Apply the function you implemented to the dataset as a whole.

In [9]:
ds_sft = smoltalk_simplified.map(format_input_output)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Then verify that the dataset now contains the new fields you created.

In [10]:
ds_sft['train'][0]

{'messages': [{'content': "You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.",
   'role': 'system'},
  {'content': 'Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.',
   'role': 'user'},
  {'content': 'The chef made more food after the restaurant ran out.',
   'role': 'assistant'}],
 'source': 'explore-instruct-rewriting',
 'prompt': "<|im_start|>system\nYou are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.<|im_end|>\n<|im_start|>user\nRearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.<|im_end|>\n<|im_start|>assistant\n",
 'response': 'The chef made more food after the restaurant ran out.<|im_end|>\n'}

### ⚙&nbsp; Task 1.3: Tokenizing the dataset

We will now prepare the format required by the HuggingFace Trainer.

We first load the tokenizer for our selected model:

In [11]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Write a function `tokenize_helper` that takes an example (using the prompt/response format from the previous step) and produces the following three results:

- `input_ids`: the integer token ids of the concatenated prompt and response;
- `labels`: a list of the same length as `input_ids`, where the response token ids are the same, but where the prompt token ids have all been replaced by the loss masking identifier -100.
- `attention_mask`: the attention mask. This should just be a list of the same length as the other two lists, with all items set to 1.

The reason why `input_ids` and `labels` are different is that
we do not want to compute the training loss for tokens that appear in the user's input. We want to train the model to generate output *conditionally*: based on a prompt. But why the magic number -100? This is the number used by default in PyTorch's [`CrossEntropyLoss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) to indicate an item that should be excluded in loss computations. (This issue was also mentioned in [Assignment 1](https://liu-nlp.ai/dl4nlp/units/a1_1.html#task-4.1-implementing-the-trainer).)

In [12]:
# SmolLM2's tokenizer ships without a pad token; reuse <|im_end|> for padding
# (we will mask all padding positions in the labels with -100 anyway).
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


def tokenize_helper(example):
    """Tokenize prompt+response into one sequence; mask the prompt tokens in labels.

    Returns input_ids, attention_mask (all ones — padding is added later by the
    collator), and labels where prompt positions are -100 so they are excluded
    from the loss.
    """
    prompt = example["prompt"]
    response = example["response"]

    # Encode separately so we know how many tokens belong to the prompt.
    # add_special_tokens=False because our format already contains the
    # special tokens we need (<|im_start|>, <|im_end|>).
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    response_ids = tokenizer(response, add_special_tokens=False)["input_ids"]

    input_ids = prompt_ids + response_ids
    attention_mask = [1] * len(input_ids)
    # Mask the prompt: only response tokens contribute to the loss.
    labels = [-100] * len(prompt_ids) + list(response_ids)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


In [13]:
tokenized_ds_sft = ds_sft.map(
    tokenize_helper,
    remove_columns=ds_sft["train"].column_names,
)
print(tokenized_ds_sft)
print("\nExample tokenized record (train[0]):")
print({k: (v[:30] if isinstance(v, list) else v) for k, v in tokenized_ds_sft["train"][0].items()})


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 400
    })
})

Example tokenized record (train[0]):
{'input_ids': [1, 9690, 198, 2683, 359, 354, 5646, 298, 12021, 11173, 30, 1206, 523, 325, 2711, 351, 253, 1694, 284, 346, 737, 288, 34013, 357, 2289, 288, 260, 2914, 506, 6388], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]}


As above, apply the function you implemented to the dataset using `map`. This will add the three new fields to the dataset.


## Part 2: Evaluation of the baseline model

As a first step, we will see how well the *baseline* model performs: that is, a model that has not been trained to follow instructions.

### ⚙&nbsp; Task 2.1: Preparing for evaluation

In this section, we set up a few utilities we will need to complete our training and evaluation infrastructure. These utilities will be given and you don't need to modify anything.

The first piece we need is a *collator*: that is, a tool that takes a number of instances and creates PyTorch tensors for a training batch. To make the batch fit into rectangular tensors, padding tokens will be added.

In [14]:
def data_collator(batch):
    """
    Create a custom collate function for causal language modeling.

    Args:
        batch: List of examples, each with 'input_ids', 'attention_mask', 'labels'
        tokenizer: Tokenizer with pad_token_id
    """

    input_ids_list = [torch.tensor(example["input_ids"], dtype=torch.long) for example in batch]
    attention_masks_list = [torch.tensor(example["attention_mask"], dtype=torch.long) for example in batch]
    labels_list = [torch.tensor(example['labels'], dtype=torch.long) for example in batch]

    # Find max length in this batch
    max_len = max(x.size(0) for x in input_ids_list)

    # Helper pad function
    def pad_to_max(x_list, pad_value):
        padded = []
        for x in x_list:
            pad_len = max_len - x.size(0)
            if pad_len > 0:
                pad_tensor = torch.full((pad_len,), pad_value, dtype=x.dtype)
                x = torch.cat([x, pad_tensor], dim=0)
            padded.append(x)
        return torch.stack(padded, dim=0)

    # Use tokenizer.pad_token_id for inputs, 0 for attention_mask, -100 for labels
    pad_id = tokenizer.pad_token_id

    batch_input_ids = pad_to_max(input_ids_list, pad_value=pad_id)
    batch_attention_mask = pad_to_max(attention_masks_list, pad_value=0)
    batch_labels = pad_to_max(labels_list, pad_value=-100)

    batch = {
            "input_ids": batch_input_ids,
            "attention_mask": batch_attention_mask,
            "labels": batch_labels,
        }
    return batch

The second utility we need is an evaluator. We will use the **ROUGE-L** metric, which computes the longest common subsequence between the model's output and the gold-standard answer. You can read about ROUGE-L here: https://en.wikipedia.org/wiki/ROUGE_(metric)

When using the ROUGE-L metric in a Trainer, we need to wrap it in an object defined as follows:

In [15]:
import evaluate

class RougeMetricComputer:
    """
    Stateful metric for batch_eval_metrics=True.

    It:
      - accumulates predictions and references across batches
      - computes ROUGE-L once at the end (compute_result=True)
    """

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.rouge = evaluate.load("rouge")
        self.all_predictions = []
        self.all_references = []

    def __call__(self, eval_pred, compute_result=False):
        """Accumulate predictions and compute at the end."""

        logits, labels = eval_pred
        pred_ids = logits.argmax(axis=-1)

        # Collect decoded answer-span text from each example in the batch
        for p, lbl in zip(pred_ids, labels):
            mask = lbl != -100
            if mask.sum() == 0:
                continue

            ref_ids = lbl[mask]
            pred_ids_filtered = p[mask]

            ref_text = self.tokenizer.decode(ref_ids, skip_special_tokens=True)
            pred_text = self.tokenizer.decode(
                pred_ids_filtered, skip_special_tokens=True,
                eos_token_id=self.tokenizer.vocab['<|im_end|>']
            )

            self.all_references.append(ref_text.strip())
            self.all_predictions.append(pred_text.strip())

        # Only compute at the very end of eval
        if compute_result:
            if len(self.all_references) > 0:
                scores = self.rouge.compute(
                    predictions=self.all_predictions,
                    references=self.all_references,
                )

                # Clear accumulated data for next eval call
                self.all_predictions = []
                self.all_references = []
                return {"rougeL": scores["rougeL"]}
            else:
                return {}
        else:
            return {}

compute_metrics = RougeMetricComputer(tokenizer)


Finally, we make a function that sets up a [`Trainer`](https://huggingface.co/docs/transformers/main_classes/trainer).

In [16]:
from transformers import Trainer
from transformers.trainer_callback import ProgressCallback

def make_trainer(model, training_args):
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds_sft["train"],
        eval_dataset=tokenized_ds_sft["test"],
        compute_metrics=compute_metrics,
        data_collator=data_collator,
    )
    trainer.callback_handler.callbacks = [
        cb for cb in trainer.callback_handler.callbacks
        if type(cb).__name__ != "NotebookProgressCallback"
    ]
    trainer.add_callback(ProgressCallback)
    return trainer


### 🎓&nbsp; Task 2.2: Evaluating the pre-trained model

Now, we have all the pieces to evaluate our baseline model that has not been instruction-tuned.

The following code will compute the loss on the test set as well as the ROUGE-L score. You will later compare these scores to the models that you train.

Why do you think the ROUGE-L score is as high as it is, even without any training for instruction-following?

In [17]:
from transformers import TrainingArguments
from transformers import AutoModelForCausalLM
import json
import time

print("\n" + "=" * 80)
print("EVALUATING PRETRAINED MODEL")
print("=" * 80)

pretrained_model = AutoModelForCausalLM.from_pretrained(model_name_or_path).to(DEVICE)

pretrained_eval_args = TrainingArguments(
    output_dir="./out_pretrained_eval",
    eval_strategy="no",
    per_device_eval_batch_size=1,
    bf16=USE_BF16, fp16=False,
    report_to="none",
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
    use_cpu=(DEVICE == "cpu"),
)

pretrained_trainer = make_trainer(pretrained_model, pretrained_eval_args)

t0 = time.perf_counter()
pretrained_eval_metrics = pretrained_trainer.evaluate()
pretrained_eval_time = time.perf_counter() - t0

pretrained_eval_loss = float(pretrained_eval_metrics["eval_loss"])
pretrained_rougeL = pretrained_eval_metrics.get("eval_rougeL", None)

print(f"\nPRETRAINED EVAL TIME: {pretrained_eval_time:.1f}s")
print("PRETRAINED EVAL METRICS:")
print(json.dumps(pretrained_eval_metrics, indent=2))



EVALUATING PRETRAINED MODEL


model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

/Users/ewishei/myClaudeCode/tyro/data/projects/dl4nlp/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


  0%|          | 0/400 [00:00<?, ?it/s]


PRETRAINED EVAL TIME: 43.2s
PRETRAINED EVAL METRICS:
{
  "eval_loss": 2.4808194637298584,
  "eval_model_preparation_time": 0.0019,
  "eval_rougeL": 0.5637525981785088,
  "eval_runtime": 43.2076,
  "eval_samples_per_second": 9.258,
  "eval_steps_per_second": 9.258,
  "epoch": 0
}


**Why is ROUGE-L non-trivial even before SFT?**

The pretrained model has never seen instruction/response pairs in this format, but ROUGE-L measures *longest common subsequence* between the predicted token sequence and the gold response — it credits any tokens that appear in roughly the right order, even if the surrounding text is wrong. Several things inflate it:

1. **Teacher forcing during eval.** The Trainer feeds the *gold* prompt+response through the model and we read off `argmax(logits)` at every position. At the second-to-last position the model already sees the entire gold response, so its top-1 prediction is essentially "shift the gold answer left by one". That gives a high token-overlap with the gold response by construction.
2. **High-frequency tokens.** Stop-words, punctuation, common verbs, and the `<|im_end|>` marker are easy for the LM to predict and overlap heavily with any reference.
3. **General pretrained competence.** SmolLM2 has already seen vast amounts of natural English; predicting the next plausible word at each position gives partial credit even without instruction following.

So this baseline tells us how well the *language modeling head* alone matches gold answers, not how well a model would do at *generating* answers from scratch. Real instruction-following only shows up in the qualitative free-running generation in Task 4.4.


## Part 3: Supervised fine-tuning



### 🎓&nbsp; Task 3.1: Training the full model

Next, we train the pre-trained model using SFT over all the parameters, then calculate the metrics and outputs to evaluate how well it follows instructions.

How do the results differ from those in the previous step?

In [18]:
baseline_training_args = TrainingArguments(
    output_dir="./out_full_sft",
    eval_strategy="epoch",
    logging_steps=2000,
    save_strategy="no",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    bf16=USE_BF16, fp16=False,
    report_to="none",
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
    use_cpu=(DEVICE == "cpu"),
    learning_rate=5e-5,
)

base_model = AutoModelForCausalLM.from_pretrained(model_name_or_path).to(DEVICE)

baseline_trainer = make_trainer(base_model, baseline_training_args)

print("\n" + "=" * 80)
print("FULL SUPERVISED FINE-TUNING")
print("=" * 80)

t0 = time.perf_counter()
baseline_train_result = baseline_trainer.train()
baseline_train_time = time.perf_counter() - t0

print(f"\nFULL SFT TRAIN TIME: {baseline_train_time:.1f}s")
print("Training metrics:", baseline_train_result.metrics)

baseline_eval_metrics = baseline_trainer.evaluate()
baseline_eval_loss = float(baseline_eval_metrics["eval_loss"])
baseline_rougeL = baseline_eval_metrics.get("eval_rougeL", None)

print("\nFULL SFT EVAL METRICS:")
print(json.dumps(baseline_eval_metrics, indent=2))


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]


FULL SUPERVISED FINE-TUNING


  0%|          | 0/5000 [00:00<?, ?it/s]

{'loss': '1.578', 'grad_norm': '6.438', 'learning_rate': '3.001e-05', 'epoch': '0.4'}
{'loss': '1.388', 'grad_norm': '10.38', 'learning_rate': '1.001e-05', 'epoch': '0.8'}


  0%|          | 0/400 [00:00<?, ?it/s]

{'eval_loss': '1.414', 'eval_rougeL': '0.635', 'eval_runtime': '30.31', 'eval_samples_per_second': '13.2', 'eval_steps_per_second': '13.2', 'epoch': '1'}
{'train_runtime': '1278', 'train_samples_per_second': '3.912', 'train_steps_per_second': '3.912', 'train_loss': '1.452', 'epoch': '1'}

FULL SFT TRAIN TIME: 1278.2s
Training metrics: {'train_runtime': 1277.9693, 'train_samples_per_second': 3.912, 'train_steps_per_second': 3.912, 'total_flos': 293276383198848.0, 'train_loss': 1.4518525390625, 'epoch': 1.0}


  0%|          | 0/400 [00:00<?, ?it/s]


FULL SFT EVAL METRICS:
{
  "eval_loss": 1.4141119718551636,
  "eval_rougeL": 0.6349575909193921,
  "eval_runtime": 28.8516,
  "eval_samples_per_second": 13.864,
  "eval_steps_per_second": 13.864,
  "epoch": 1.0
}


### ⚙&nbsp; Task 3.3: Counting the number of trainable parameters

Define a function `num_trainable_parameters` that computes the number of floating-point numbers that a given model will update during training.

**Hints**:
- For a PyTorch module `m`, you can use `m.parameters()` to access its parameter tensors.
- However, you should only include parameter tensors where the flag `requires_grad` is True.


In [19]:
def num_trainable_parameters(model):
    """Count number of trainable parameters (those with requires_grad=True)."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


In [20]:
n_full = num_trainable_parameters(base_model)
print(f"Trainable parameters in full SFT model: {n_full:,}")
print(f"  ≈ {n_full / 1e6:.1f}M")


Trainable parameters in full SFT model: 134,515,008
  ≈ 134.5M


Apply this function to the SFT-trained model and check that the result makes sense.

## Part 4: Parameter-efficient fine-tuning

In the last section of this assignment, we will use LoRA to train the model in a more parameter-efficient manner. You may want to prepare by reading  by [the paper by Hu et al. (2021)](https://arxiv.org/pdf/2106.09685) and the teaching material provided for this course.

### ⚙&nbsp; Task 4.1: Utilities for modifying models

Define a function `extract_lora_targets` that extracts the relevant linear layers from all Transformer blocks in your selected LLM.
It is up to you to decide what layers to select; in the experiments described in the original LoRA paper, the query and value projection matrices were fine-tuned with LoRA, while all other layers were left unchanged.
Return a dictionary that maps the component name to the corresponding linear layer.

As we saw earlier (in Assignment 2 and elsewhere), a Transformer model consists of a hierarchy of nested submodules. Each of these can be addressed by a fully-qualified string name. You can use get_submodule() to retrieve a layer by a string name. This name depends on the model you have selected. For instance, in the `SmolLM2-135M` model, `'model.layers.0.self_attn.q_proj'`
 refers to the query projection in Transformer layer 0.

It is OK to hard-code this part, so that you just enumerate the layers you want to extract. Alternatively, use a utility such as `model.named_modules()` to iterate through the model's layers.

In [21]:
def extract_lora_targets(model, target_names=("q_proj", "k_proj", "v_proj", "o_proj")):
    """Return {fully_qualified_name: nn.Linear} for every attention projection.

    SmolLM2 layers are named `model.layers.{i}.self_attn.{q,k,v,o}_proj`.
    We use named_modules() so this also works for any LLaMA-style architecture
    that uses the same names.
    """
    targets = {}
    for name, module in model.named_modules():
        # name endswith one of the target projection names
        if any(name.endswith("." + t) for t in target_names):
            if isinstance(module, nn.Linear):
                targets[name] = module
    return targets


We also need a convenience function that puts layers back into a model. The following function does the trick. The `named_layers` argument uses the same format as returned by `extract_lora_targets`.

In [22]:
def replace_layers(model, named_layers):
    """
    Replace submodules in `model` by name.
    """
    for name, layer in named_layers.items():
        components = name.split(".")
        submodule = model
        for comp in components[:-1]:
            submodule = getattr(submodule, comp)
        setattr(submodule, components[-1], layer)
    return model

### 🎓&nbsp; Task 4.2: Implementing the LoRA layer

To implement the LoRA approach, we define a new type of layer that will be used as a drop-in replacement for a regular linear layer.

In [the paper by Hu et al. (2021)](https://arxiv.org/pdf/2106.09685), the structure is presented visually in Figure 1, and equation (3) shows the same idea.

Start from the following skeleton and fill in the missing pieces:


In [23]:
import torch.nn as nn
import math


class LoRALayer(nn.Module):
    """Drop-in replacement for nn.Linear that adds a low-rank update.

    Implements   y = W x + (alpha / r) * B (A x)
    where A in R^{r x in_features}, B in R^{out_features x r}.

    Following Hu et al. (2021):
    - The original weight `W` (and its bias) is FROZEN. Only A and B are trained.
    - A is initialized with Kaiming uniform (small Gaussian-ish) and B is
      initialized to zero, so the residual update starts at exactly 0 and
      training begins from the unmodified pretrained behaviour.
    """

    def __init__(self, W, r, alpha):
        super().__init__()
        if not isinstance(W, nn.Linear):
            raise TypeError(f"LoRALayer expects nn.Linear, got {type(W)}")
        self.W = W
        # Freeze the wrapped pretrained weight and bias.
        for p in self.W.parameters():
            p.requires_grad = False

        in_features = W.in_features
        out_features = W.out_features
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        # Trainable low-rank factors. Match dtype/device of the wrapped layer.
        device = W.weight.device
        dtype = W.weight.dtype
        self.A = nn.Parameter(torch.empty(r, in_features, device=device, dtype=dtype))
        self.B = nn.Parameter(torch.zeros(out_features, r, device=device, dtype=dtype))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def forward(self, x):
        # Frozen base path + low-rank delta, scaled by alpha/r.
        base = self.W(x)
        delta = (x @ self.A.T) @ self.B.T
        return base + self.scaling * delta


Here, `W` is the linear layer we are fine-tuning, while `r` and `alpha` are hyperparameters described in section 4.1. of the paper. The `r` parameter controls the parameter efficiency: by setting it to a low value, we save memory but make a rougher approximation. The `alpha` parameter is a scaling factor.

### 🎓&nbsp; Task 4.3: Fine-tuning with LoRA

Set up a model where you replace the four linear layers in attention blocks (query, key, value, and output) with LoRA layers. Use the following steps:
- First use `extract_lora_targets` to get the relevant linear layers.
- Each of the linear layers in the returned dictionary should be wrapped inside a LoRA layer.
- Then use `replace_layers` to put them back into the model.

Train this model and compare the training speed, metrics, and outputs to the results from Part 3.

Apply your parameter counting function (`num_trainable_parameters`) to this model, compare the results to those in Part 3, and make sure that these results correspond to your expectations.


In [24]:
LORA_R = 8
LORA_ALPHA = 16

# Fresh copy of the pretrained model (so we don't reuse the full-SFT weights).
lora_model = AutoModelForCausalLM.from_pretrained(model_name_or_path).to(DEVICE)

# Freeze EVERYTHING first; LoRALayer.__init__ leaves A, B trainable.
for p in lora_model.parameters():
    p.requires_grad = False

# Find the q/k/v/o projections, wrap each in a LoRALayer, plug back in.
targets = extract_lora_targets(lora_model)
print(f"Wrapping {len(targets)} linear layers with LoRA (r={LORA_R}, alpha={LORA_ALPHA}).")
wrapped = {name: LoRALayer(layer, r=LORA_R, alpha=LORA_ALPHA) for name, layer in targets.items()}
replace_layers(lora_model, wrapped)
lora_model.to(DEVICE)

n_lora = num_trainable_parameters(lora_model)
print(f"Trainable parameters in LoRA model: {n_lora:,}")
print(f"  ≈ {n_lora / 1e6:.3f}M  ({100 * n_lora / n_full:.2f}% of full SFT model)")


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Wrapping 120 linear layers with LoRA (r=8, alpha=16).
Trainable parameters in LoRA model: 921,600
  ≈ 0.922M  (0.69% of full SFT model)


In [25]:
lora_training_args = TrainingArguments(
    output_dir="./out_lora_sft",
    eval_strategy="epoch",
    logging_steps=2000,
    save_strategy="no",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    bf16=USE_BF16, fp16=False,
    report_to="none",
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
    use_cpu=(DEVICE == "cpu"),
    learning_rate=2e-4,  # LoRA usually wants a higher LR than full SFT
)

lora_trainer = make_trainer(lora_model, lora_training_args)

print("\n" + "=" * 80)
print("LoRA FINE-TUNING")
print("=" * 80)

t0 = time.perf_counter()
lora_train_result = lora_trainer.train()
lora_train_time = time.perf_counter() - t0

print(f"\nLoRA TRAIN TIME: {lora_train_time:.1f}s   (full SFT was {baseline_train_time:.1f}s)")
print("Training metrics:", lora_train_result.metrics)

lora_eval_metrics = lora_trainer.evaluate()
lora_eval_loss = float(lora_eval_metrics["eval_loss"])
lora_rougeL = lora_eval_metrics.get("eval_rougeL", None)

print("\nLoRA EVAL METRICS:")
print(json.dumps(lora_eval_metrics, indent=2))



LoRA FINE-TUNING


  0%|          | 0/5000 [00:00<?, ?it/s]

{'loss': '1.566', 'grad_norm': '0.8945', 'learning_rate': '0.00012', 'epoch': '0.4'}
{'loss': '1.381', 'grad_norm': '1.391', 'learning_rate': '4.004e-05', 'epoch': '0.8'}


  0%|          | 0/400 [00:00<?, ?it/s]

{'eval_loss': '1.392', 'eval_rougeL': '0.638', 'eval_runtime': '38.37', 'eval_samples_per_second': '10.43', 'eval_steps_per_second': '10.43', 'epoch': '1'}
{'train_runtime': '801.1', 'train_samples_per_second': '6.242', 'train_steps_per_second': '6.242', 'train_loss': '1.446', 'epoch': '1'}

LoRA TRAIN TIME: 801.9s   (full SFT was 1278.2s)
Training metrics: {'train_runtime': 801.0788, 'train_samples_per_second': 6.242, 'train_steps_per_second': 6.242, 'total_flos': 295821342891648.0, 'train_loss': 1.4464465087890626, 'epoch': 1.0}


  0%|          | 0/400 [00:00<?, ?it/s]


LoRA EVAL METRICS:
{
  "eval_loss": 1.3920471668243408,
  "eval_rougeL": 0.6379868374889938,
  "eval_runtime": 45.6859,
  "eval_samples_per_second": 8.755,
  "eval_steps_per_second": 8.755,
  "epoch": 1.0
}


In [26]:
# Summary comparison.
def fmt(v):
    return "n/a" if v is None else f"{v:.4f}"

print("=" * 70)
print(f"{'Model':<18}{'eval_loss':>12}{'rougeL':>12}{'trainable':>15}{'train_s':>10}")
print("-" * 70)
print(f"{'Pretrained':<18}{fmt(pretrained_eval_loss):>12}{fmt(pretrained_rougeL):>12}{'-':>15}{'-':>10}")
print(f"{'Full SFT':<18}{fmt(baseline_eval_loss):>12}{fmt(baseline_rougeL):>12}{n_full:>15,}{baseline_train_time:>10.1f}")
print(f"{'LoRA (r=8)':<18}{fmt(lora_eval_loss):>12}{fmt(lora_rougeL):>12}{n_lora:>15,}{lora_train_time:>10.1f}")
print("=" * 70)


Model                eval_loss      rougeL      trainable   train_s
----------------------------------------------------------------------
Pretrained              2.4808      0.5638              -         -
Full SFT                1.4141      0.6350    134,515,008    1278.2
LoRA (r=8)              1.3920      0.6380        921,600     801.9


### 🎓&nbsp; Task 4.4: Qualitative inspection

Run the three models interactively on some examples of your own choice (either taken from the training or test sets, or created by yourself). The convenience function below can be of use, but you need to complete it by using the prompt format you defined in Task 1.2.

Do your models seem to have learned the instruction-following behavior (at least to some extent)? Do they respond to user queries sensibly?

The quality we see here will depend on your choice of base model as well as how much you trained it.

In [27]:
@torch.no_grad()
def generate_response(model, user_text, system_text=None, max_new_tokens=120,
                      do_sample=False, temperature=0.7, top_p=0.9):
    """Render a user message into our chat template and let the model generate."""
    msgs = []
    if system_text is not None:
        msgs.append({"role": "system", "content": system_text})
    msgs.append({"role": "user", "content": user_text})
    # Use format_input_output but swap in a dummy assistant turn so we get a clean prompt.
    fake = {"messages": msgs + [{"role": "assistant", "content": ""}]}
    prompt = format_input_output(fake)["prompt"]

    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature if do_sample else 1.0,
        top_p=top_p if do_sample else 1.0,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=im_end_id,
    )
    new_tokens = out[0, inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(new_tokens, skip_special_tokens=False)
    # Trim at the first <|im_end|> if present.
    if "<|im_end|>" in text:
        text = text.split("<|im_end|>")[0]
    return text.strip()


In [28]:
test_prompts = [
    "What is the capital of Sweden?",
    "Write a haiku about autumn.",
    "Explain the concept of overfitting in one sentence.",
    "List three uses of LoRA in machine learning.",
]

for q in test_prompts:
    print("=" * 78)
    print(f"USER: {q}")
    print("-" * 78)
    print("[Pretrained]")
    print(generate_response(pretrained_model, q))
    print("\n[Full SFT]")
    print(generate_response(base_model, q))
    print("\n[LoRA]")
    print(generate_response(lora_model, q))
    print()


USER: What is the capital of Sweden?
------------------------------------------------------------------------------
[Pretrained]
What is the capital of Sweden?
What is the capital of Sweden?
What is the capital of Sweden?
What is the capital of Sweden?
What is the capital of Sweden?
What is the capital of Sweden?
What is the capital of Sweden?
What is the capital of Sweden?
What is the capital of Sweden?
What is the capital of Sweden?
What is the capital of Sweden?
What is the capital of Sweden?
What is the capital of Sweden?
What is the capital of Sweden?
What is the capital of Sweden?

[Full SFT]
The capital of Sweden is Stockholm.

[LoRA]
The capital of Sweden is Stockholm.

USER: Write a haiku about autumn.
------------------------------------------------------------------------------
[Pretrained]
Write a haiku about autumn.
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant
Assistant